## 🗣️ CTI-Bench MCQ Evaluation

This notebook evaluates Qwen2.5 models (base & finetuned) on the CTI-Bench MCQ dataset using **greedy decoding**.

In [1]:
import os
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig, get_peft_model, set_peft_model_state_dict
import torch
from datasets import load_dataset
from huggingface_hub import login

# Authenticate with HuggingFace Hub (set your token here or via HF_TOKEN env var)
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged in to HuggingFace Hub")
else:
    print("⚠️  No HF_TOKEN set. Run: export HF_TOKEN=hf_xxx  (in your WSL terminal before starting Jupyter)")

/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged in to HuggingFace Hub


### 📋 Model Registry

All available models. Pick one by setting `SELECTED_MODEL` in the next cell.

In [2]:
# ========================================
# 📋 MODEL REGISTRY
# ========================================
# Each entry: {"path": ..., "params": ..., "type": "base" or "finetuned", "source": "remote" or "local"}

MODEL_REGISTRY = {
    # ---- Base models (HuggingFace remote) ----
    "Qwen2.5-0.5B-base": {
        "path": "Qwen/Qwen2.5-0.5B-Instruct",
        "params": "0.5B",
        "type": "base",
        "source": "remote",
    },
    "Qwen2.5-1.5B-base": {
        "path": "Qwen/Qwen2.5-1.5B-Instruct",
        "params": "1.5B",
        "type": "base",
        "source": "remote",
    },
    "Qwen2.5-7B-base": {
        "path": "Qwen/Qwen2.5-7B-Instruct",
        "params": "7B",
        "type": "base",
        "source": "remote",
    },

    # ---- Finetuned models (local) ----
    "Qwen2.5-0.5B-finetuned": {
        "path": "models/finetuned/Qwen2.5-0.5B-Instruct-CYSECFINETUNED",
        "params": "0.5B",
        "type": "finetuned",
        "source": "local",
    },
    "Qwen2.5-1.5B-finetuned": {
        "path": "models/finetuned/Qwen2.5-1.5B-Instruct-CYSECFINETUNED",
        "params": "1.5B",
        "type": "finetuned",
        "source": "local",
    },
    "Qwen2.5-7B-finetuned": {
        "path": "models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE",
        "params": "7B",
        "type": "finetuned",
        "source": "local",  
    },
    "Qwen2.5-7B-finetuned-remake": {
        "path": "models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE",
        "params": "7B",
        "type": "finetuned",
        "source": "local",  
    },
}

# Print available models
print("Available models:")
for key, info in MODEL_REGISTRY.items():
    print(f"  - {key:30s}  ({info['source']:6s})  {info['path']}")

Available models:
  - Qwen2.5-0.5B-base               (remote)  Qwen/Qwen2.5-0.5B-Instruct
  - Qwen2.5-1.5B-base               (remote)  Qwen/Qwen2.5-1.5B-Instruct
  - Qwen2.5-7B-base                 (remote)  Qwen/Qwen2.5-7B-Instruct
  - Qwen2.5-0.5B-finetuned          (local )  models/finetuned/Qwen2.5-0.5B-Instruct-CYSECFINETUNED
  - Qwen2.5-1.5B-finetuned          (local )  models/finetuned/Qwen2.5-1.5B-Instruct-CYSECFINETUNED
  - Qwen2.5-7B-finetuned            (local )  models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE
  - Qwen2.5-7B-finetuned-remake     (local )  models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE


### ⚙️ Configuration

Change `SELECTED_MODEL` to pick a model.  
Set `SUBSET_SIZE = None` to evaluate on all 2500 questions, or an integer for a quick test.

In [3]:
# ========================================
# 🔧 CONFIGURATION
# ========================================

# Pick a model from MODEL_REGISTRY
SELECTED_MODEL = "Qwen2.5-7B-finetuned-remake"  # <-- CHANGE THIS

# Dataset subset: None = all 2500, or an integer (e.g. 100)
SUBSET_SIZE = None

# System Prompt
SYSTEM_PROMPT = "You are a cybersecurity expert specializing in cyberthreat intelligence."

# Generation Parameters (greedy decoding)
MAX_TOKENS = 2048
SEED = 42

# Hardware
USE_GPU = True
LOAD_IN_8BIT = True 

# ========================================
# Auto-detect environment
# ========================================
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# Resolve model info
model_info = MODEL_REGISTRY[SELECTED_MODEL]
MODEL_PATH = model_info["path"]
MODEL_PARAMS = model_info["params"]
MODEL_TYPE = model_info["type"]
MODEL_SOURCE = model_info["source"]

# Auto-format model name: Qwen2.5-{params}-{type}
MODEL_NAME = f"Qwen2.5-{MODEL_PARAMS}-{MODEL_TYPE}"

print(f"✓ Configuration loaded")
print(f"  Selected  : {SELECTED_MODEL}")
print(f"  Path      : {MODEL_PATH}")
print(f"  Source    : {MODEL_SOURCE}")
print(f"  Name (out): {MODEL_NAME}")
print(f"  Subset    : {'ALL (2500)' if SUBSET_SIZE is None else SUBSET_SIZE}")
print(f"  Env       : {'Google Colab' if IS_COLAB else 'Local'}")
print(f"  GPU       : {USE_GPU}")
print(f"  8-bit     : {LOAD_IN_8BIT}")
print(f"  Decoding  : Greedy (do_sample=False)")

✓ Configuration loaded
  Selected  : Qwen2.5-7B-finetuned-remake
  Path      : models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE
  Source    : local
  Name (out): Qwen2.5-7B-finetuned
  Subset    : ALL (2500)
  Env       : Local
  GPU       : True
  8-bit     : True
  Decoding  : Greedy (do_sample=False)


### 📦 Dataset Prep

In [4]:
ds = load_dataset("AI4Sec/cti-bench", "cti-mcq")
data = ds['test'] if 'test' in ds else ds['train']

# Apply subset
if SUBSET_SIZE is not None:
    data_subset = data.select(range(min(SUBSET_SIZE, len(data))))
else:
    data_subset = data

print(f"Total examples in dataset : {len(data)}")
print(f"Examples to evaluate      : {len(data_subset)}")
print(f"Column names: {data.column_names}")
print()
print("First example:")
print(data[0])

Total examples in dataset : 2500
Examples to evaluate      : 2500
Column names: ['URL', 'Question', 'Option A', 'Option B', 'Option C', 'Option D', 'Prompt', 'GT']

First example:
{'URL': 'https://attack.mitre.org/techniques/T1548/', 'Question': "Which of the following mitigations involves preventing applications from running that haven't been downloaded from legitimate repositories?", 'Option A': 'Audit', 'Option B': 'Execution Prevention', 'Option C': 'Operating System Configuration', 'Option D': 'User Account Control', 'Prompt': "You are given a multiple-choice question (MCQ) from a Cyber Threat Intelligence (CTI) knowledge benchmark dataset. Your task is to choose the best option among the four provided. Return your answer as a single uppercase letter: A, B, C, or D.  **Question:** Which of the following mitigations involves preventing applications from running that haven't been downloaded from legitimate repositories?  **Options:** A) Audit B) Execution Prevention C) Operating Sys

### 🌐 Load Remote Model (HuggingFace)

Run this cell if `MODEL_SOURCE == "remote"` (base models from HuggingFace Hub).

In [ ]:
# ========================================
# 🌐 LOAD REMOTE MODEL (HuggingFace)
# ========================================
assert MODEL_SOURCE == "remote", (
    f"This cell is for remote models. Current model '{SELECTED_MODEL}' is '{MODEL_SOURCE}'. "
    f"Use the 'Load Local Model' cell instead."
)

print(f"Loading remote model: {MODEL_PATH}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Device
if USE_GPU and torch.cuda.is_available():
    device = "cuda"
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("✓ Using CPU")

# Quantization config
quant_config = None
if LOAD_IN_8BIT and device == "cuda":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    print("✓ Using 8-bit quantization")

# Model
if quant_config:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    print("✓ Model loaded in 8-bit mode")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    model = model.to(device)
    print(f"✓ Model loaded on {device}")

print(f"✓ Remote model ready: {MODEL_NAME}")

### 💾 Load Local Model (Finetuned)

Run this cell if `MODEL_SOURCE == "local"` (finetuned models from disk).

In [5]:
# ========================================
# 💾 LOAD LOCAL MODEL (Finetuned)
# ========================================
assert MODEL_SOURCE == "local", (
    f"This cell is for local models. Current model '{SELECTED_MODEL}' is '{MODEL_SOURCE}'. "
    f"Use the 'Load Remote Model' cell instead."
)

# Resolve path based on environment
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    local_model_path = os.path.join('/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM', MODEL_PATH)
else:
    # Try multiple candidate repo roots (cwd, remote server, local WSL)
    _CANDIDATE_ROOTS = [
        os.getcwd(),
        "/mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM",
        "/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM",
    ]
    local_model_path = None
    for _root in _CANDIDATE_ROOTS:
        _candidate = os.path.join(_root, MODEL_PATH)
        if os.path.exists(_candidate):
            local_model_path = _candidate
            break
    if local_model_path is None:
        raise FileNotFoundError(
            f"Model path '{MODEL_PATH}' not found in any of:\n"
            + "\n".join(f"  - {r}" for r in _CANDIDATE_ROOTS)
        )

print(f"Loading local model: {local_model_path}")
print(f"  Contents: {os.listdir(local_model_path)}")

# Check if this is a LoRA adapter (has adapter_config.json) or a full model (has config.json)
is_lora_adapter = os.path.exists(os.path.join(local_model_path, "adapter_config.json"))
is_full_model = os.path.exists(os.path.join(local_model_path, "config.json"))
print(f"  LoRA adapter: {is_lora_adapter} | Full model: {is_full_model}")

# Device
if USE_GPU and torch.cuda.is_available():
    device = "cuda"
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("✓ Using CPU")

# Quantization config
quant_config = None
if LOAD_IN_8BIT and device == "cuda":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    print("✓ Using 8-bit quantization")

if is_lora_adapter:
    # ---- LoRA adapter: load base model first, then apply adapter ----
    peft_config = PeftConfig.from_pretrained(local_model_path) # <---
    base_model_id = peft_config.base_model_name_or_path
    print(f"✓ Detected LoRA adapter")
    print(f"  Base model (from adapter config): {base_model_id}")

    # If base_model_id is a local path that doesn't exist, resolve it
    if not os.path.exists(base_model_id) and ("/" in base_model_id or "\\" in base_model_id):
        _base_dirname = os.path.basename(base_model_id.rstrip("/\\"))
        print(f"Base Directory Name : {_base_dirname}")
        _resolved = None
        if not IS_COLAB:
            for _root in _CANDIDATE_ROOTS:
                _try_path = os.path.join(_root, "models", "base", _base_dirname)
                if os.path.exists(_try_path):
                    _resolved = _try_path
                    break
        if _resolved:
            base_model_id = _resolved
            print(f"  → Found base model locally: {base_model_id}")
        else:
            # Derive HuggingFace repo ID from the directory name
            base_model_id = f"Qwen/{_base_dirname}"
            print(f"  → Resolved to HF repo: {base_model_id}")

    # Load base model
    if quant_config:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=quant_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        base_model = base_model.to(device)
    print(f"✓ Base model loaded")

    # Fix peft_config in memory so it uses the resolved base model ID
    peft_config.base_model_name_or_path = base_model_id

    # Apply LoRA adapter — use get_peft_model + manual weight loading
    # to avoid PeftModel.from_pretrained's HF Hub repo-id validation on local paths
    model = get_peft_model(base_model, peft_config)

    # Load adapter weights from local files
    _safetensors_path = os.path.join(local_model_path, "adapter_model-001.safetensors")
    _bin_path = os.path.join(local_model_path, "adapter_model.bin")
    if os.path.exists(_safetensors_path):
        from safetensors.torch import load_file
        _adapter_state = load_file(_safetensors_path)
        print(f"  Loaded adapter weights from adapter_model-001.safetensors")
    elif os.path.exists(_bin_path):
        _adapter_state = torch.load(_bin_path, map_location="cpu", weights_only=True)
        print(f"  Loaded adapter weights from adapter_model.bin")
    else:
        raise FileNotFoundError(
            f"No adapter weight files found in '{local_model_path}'.\n"
            f"Expected 'adapter_model.safetensors' or 'adapter_model.bin'.\n"
            f"Files found: {os.listdir(local_model_path)}"
        )

    set_peft_model_state_dict(model, _adapter_state)
    model.eval()
    print(f"✓ LoRA adapter applied")

    # Tokenizer: try adapter dir first, fall back to base model
    has_tokenizer = os.path.exists(os.path.join(local_model_path, "tokenizer_config.json"))
    if has_tokenizer:
        tokenizer = AutoTokenizer.from_pretrained(local_model_path)
        print(f"✓ Tokenizer loaded from adapter directory")
    else:
        tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        print(f"✓ Tokenizer loaded from base model ({base_model_id})")

elif is_full_model:
    # ---- Full model (merged weights with config.json) ----
    tokenizer = AutoTokenizer.from_pretrained(local_model_path)

    if quant_config:
        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            quantization_config=quant_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        print("✓ Model loaded in 8-bit mode")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        model = model.to(device)
        print(f"✓ Model loaded on {device}")

else:
    raise ValueError(
        f"Directory '{local_model_path}' has neither adapter_config.json nor config.json.\n"
        f"Contents: {os.listdir(local_model_path)}"
    )

print(f"✓ Local model ready: {MODEL_NAME}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading local model: /mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM/models/finetuned/Qwen2.5-7B-Instruct-CYSECFINETUNED-REMAKE
  Contents: ['adapter_config.json', 'adapter_model-001.safetensors', 'added_tokens.json', 'chat_template.jinja', 'merges.txt', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
  LoRA adapter: True | Full model: False
✓ Using GPU: NVIDIA GeForce RTX 4080 SUPER
✓ Using 8-bit quantization
✓ Detected LoRA adapter
  Base model (from adapter config): /mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM/models/base/qwen2.5-7b-instruct-unsloth-bnb-4bit


/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/transformers/quantizers/auto.py:259: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading weights: 100%|████████████████████████████████████████████████████████████████| 339/339 [00:59<00:00,  5.68it/s]


✓ Base model loaded


/home/aditya/documents/code/other_works/python_jupyter/jupyter_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


  Loaded adapter weights from adapter_model-001.safetensors
✓ LoRA adapter applied
✓ Tokenizer loaded from adapter directory
✓ Local model ready: Qwen2.5-7B-finetuned


### 🧠 Prediction & Evaluation

In [ ]:
def get_single_prediction(question):
    """Generate a prediction using greedy decoding."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,  # Greedy decoding
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_ids = outputs[0][input_length:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # print(f"Response: '{response}'")
    return response


def format_mcq(text):
    """Extract the A/B/C/D answer from model output."""
    last_line = text.split('\n')[-1].rstrip()

    if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
        return last_line[0]
    if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
        return last_line[-1]
    if last_line.endswith('**'):
        return last_line[-3]
    if len(last_line) == 0:
        last_line = text.split('\n')[-2].rstrip()
        if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
            return last_line[0]
        if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
            return last_line[-1]
        if last_line.endswith('**'):
            return last_line[-3]
    return ' '.join(text.split('\n'))


print("✓ Formatting functions loaded")


def run_evaluation(task, save_to_drive=True):
    """
    Run evaluation on CTI-Bench task.

    Args:
        task: One of 'cti-mcq', 'cti-rcm', 'cti-vsp', 'cti-taa'
        save_to_drive: If True and on Colab, save to Google Drive
    """
    print(f"\n{'='*60}")
    print(f"Starting evaluation: {task.upper()}")
    print(f"Model: {MODEL_NAME}")
    print(f"Dataset: AI4Sec/cti-bench")
    print(f"Subset: {len(data_subset)} examples")
    print(f"Decoding: Greedy (do_sample=False)")
    print(f"{'='*60}\n")

    # Find prompt column
    prompt_column = None
    for col in ['Prompt', 'prompt', 'question', 'input', 'text']:
        if col in data_subset.column_names:
            prompt_column = col
            break

    if prompt_column is None:
        print(f"Available columns: {data_subset.column_names}")
        raise ValueError("Could not find prompt column. Please specify manually.")

    print(f"Using column '{prompt_column}' for prompts\n")

    # Track metrics
    start_time = time.time()
    count_chars = 0
    instructions_failed = 0

    # Storage
    all_responses = []
    all_results = []
    task_type = task.split('-')[-1]

    # Process each example
    for index, example in enumerate(data_subset):
        prompt = example[prompt_column]
        try:
            output = get_single_prediction(prompt)
            count_chars += len(output)
            all_responses.append(output)

            if task_type == 'mcq':
                answer = format_mcq(output)
            else:
                raise ValueError(f'Unknown task type: {task_type}')

        except Exception as e:
            answer = 'Error'
            all_responses.append(answer)
            print(f'❌ Exception at example {index+1}: {e}')

        all_results.append(answer)
        # print(f'{index+1}/{len(data_subset)}: {answer}')

    # Metrics
    time_taken = time.time() - start_time

    print(f"\n{'='*60}")
    print("EVALUATION COMPLETE")
    print(f"{'='*60}")
    print(f"Time taken: {time_taken:.2f} seconds ({time_taken/60:.2f} minutes)")
    print(f"Characters generated: {count_chars:,}")
    print(f"Instructions failed: {instructions_failed}/{len(data_subset)}")
    print(f"Success rate: {(len(data_subset)-instructions_failed)/len(data_subset)*100:.1f}%")

    # Determine output directory
    if IS_COLAB and save_to_drive:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("\n⚠️  Google Drive not mounted. Mounting now...")
            drive.mount('/content/drive')
        output_dir = '/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/results'
    else:
        # Try multiple candidate roots for results directory
        # (prefer /mnt/d project dir over Jupyter kernel cwd)
        _CANDIDATE_ROOTS = [
            "/mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM",
            "/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM",
            os.getcwd(),
        ]
        output_dir = None
        for _root in _CANDIDATE_ROOTS:
            if os.path.isdir(_root):
                output_dir = os.path.join(_root, 'results')
                break
        if output_dir is None:
            output_dir = './results'

    os.makedirs(output_dir, exist_ok=True)

    # Build filename: cti-mcq_Qwen2.5-1.5B-finetuned[_firstN]_result.txt
    suffix = f"_first{len(data_subset)}" if SUBSET_SIZE is not None else ""
    out_response = f"{output_dir}/{task}_{MODEL_NAME}{suffix}_response.txt"
    out_result = f"{output_dir}/{task}_{MODEL_NAME}{suffix}_result.txt"

    with open(out_response, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_responses))

    with open(out_result, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_results))

    print(f"\n💾 Saving to: {output_dir}")
    print(f"\n📁 Results saved:")
    print(f"  - Full responses: {out_response}")
    print(f"  - Formatted results: {out_result}")
    print(f"\n{'='*60}\n")

    return all_results


print("✓ Evaluation function ready")

✓ Formatting functions loaded
✓ Evaluation function ready


### 🚀 Run Evaluation

In [7]:
results = run_evaluation('cti-mcq', save_to_drive=IS_COLAB)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Starting evaluation: CTI-MCQ
Model: Qwen2.5-7B-finetuned
Dataset: AI4Sec/cti-bench
Subset: 2500 examples
Decoding: Greedy (do_sample=False)

Using column 'Prompt' for prompts



Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


EVALUATION COMPLETE
Time taken: 497.60 seconds (8.29 minutes)
Characters generated: 2,500
Instructions failed: 0/2500
Success rate: 100.0%

💾 Saving to: /home/aditya/documents/code/other_works/python_jupyter/results

📁 Results saved:
  - Full responses: /home/aditya/documents/code/other_works/python_jupyter/results/cti-mcq_Qwen2.5-7B-finetuned_response.txt
  - Formatted results: /home/aditya/documents/code/other_works/python_jupyter/results/cti-mcq_Qwen2.5-7B-finetuned_result.txt


